# Multi-Digit Handwritten Number Recognition with CNN

Recognize **five-digit handwritten numbers** by combining digits from the [MNIST dataset](https://www.tensorflow.org/datasets/catalog/mnist) and training a convolutional neural network (CNN).

**Workflow:** load MNIST → explore sample images → generate five-digit examples → train/validate CNN → evaluate per-digit and exact-sequence accuracy → visualize predictions.

> **Note:** This is a cleaned and corrected version of an earlier coursework notebook. Its original objective and image-cropping strategy are retained, while the classification labels, loss function, and accuracy reporting are corrected for the five-digit task. Results depend on actually training the model; this notebook is distributed without saved outputs.

## 1. Imports and configuration

Install the dependencies in `requirements.txt` before running. A GPU is recommended, but not required.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow import keras
from tensorflow.keras import layers, regularizers

SEED = 42
DIGITS_PER_IMAGE = 5
NUM_CLASSES = 10
TRAIN_SAMPLES = 51_000
VALID_SAMPLES = 9_000
TEST_SAMPLES = 10_000
BATCH_SIZE = 128
EPOCHS = 20

keras.utils.set_random_seed(SEED)


## 2. Load and explore the MNIST dataset

MNIST contains 60,000 training images and 10,000 separate test images. Each grayscale image is 28 × 28 pixels and represents one digit (0–9). The images will be combined later to form new five-digit examples.

In [ ]:
(x_mnist_train, y_mnist_train), (x_mnist_test, y_mnist_test) = keras.datasets.mnist.load_data()

print(f"MNIST train: {x_mnist_train.shape}, labels: {y_mnist_train.shape}")
print(f"MNIST test:  {x_mnist_test.shape}, labels: {y_mnist_test.shape}")


In [ ]:
fig, axes = plt.subplots(4, 10, figsize=(13, 5))
for ax, image, label in zip(axes.flat, x_mnist_train[:40], y_mnist_train[:40]):
    ax.imshow(image, cmap="gray")
    ax.set_title(str(label), fontsize=9)
    ax.axis("off")
plt.suptitle("Sample handwritten digits from MNIST")
plt.tight_layout()
plt.show()


## 3. Generate synthetic five-digit examples

Each generated example is made by horizontally stacking five cropped MNIST digits. The crop matches the earlier coursework notebook: `image[4:26, 5:23]`, yielding a 22 × 18 digit crop and a **22 × 90** final image.

The first digit is sampled from 1–9, so every example is a five-digit number without a leading zero. The four following digits can be 0–9.

To avoid sharing source handwriting samples across training and validation, **split the original MNIST training images before composing the new images**. The official MNIST test set remains completely held out.

In [ ]:
def make_five_digit_images(images, labels, sample_count, rng):
    """Build (N, 22, 90) images and (N, 5) digit labels from MNIST.

    Images remain uint8 to reduce memory usage until normalization.
    Each digit position is sampled independently with replacement.
    """
    if len(images) != len(labels):
        raise ValueError("Images and labels must have matching lengths.")
    if images.ndim != 3 or images.shape[1:] != (28, 28):
        raise ValueError("Expected MNIST images with shape (N, 28, 28).")

    crops = images[:, 4:26, 5:23]  # (N, 22, 18)
    nonzero_indices = np.flatnonzero(labels != 0)
    if len(nonzero_indices) == 0:
        raise ValueError("At least one nonzero digit is required.")

    chosen = rng.integers(0, len(images), size=(sample_count, DIGITS_PER_IMAGE))
    chosen[:, 0] = rng.choice(nonzero_indices, size=sample_count)

    combined_images = np.concatenate(
        [crops[chosen[:, position]] for position in range(DIGITS_PER_IMAGE)],
        axis=2,
    )
    digit_labels = labels[chosen].astype(np.int64)
    return combined_images, digit_labels


source_train_x, source_valid_x, source_train_y, source_valid_y = train_test_split(
    x_mnist_train,
    y_mnist_train,
    test_size=0.15,
    random_state=SEED,
    stratify=y_mnist_train,
)

rng = np.random.default_rng(SEED)
train_images, train_labels = make_five_digit_images(
    source_train_x, source_train_y, TRAIN_SAMPLES, rng
)
valid_images, valid_labels = make_five_digit_images(
    source_valid_x, source_valid_y, VALID_SAMPLES, rng
)
test_images, test_labels = make_five_digit_images(
    x_mnist_test, y_mnist_test, TEST_SAMPLES, rng
)

assert train_images.shape == (TRAIN_SAMPLES, 22, 90)
assert train_labels.shape == (TRAIN_SAMPLES, DIGITS_PER_IMAGE)
assert np.all(train_labels[:, 0] != 0)

print(f"Train:      {train_images.shape} | labels: {train_labels.shape}")
print(f"Validation: {valid_images.shape} | labels: {valid_labels.shape}")
print(f"Test:       {test_images.shape} | labels: {test_labels.shape}")


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 4))
for ax, image, digits in zip(axes.flat, train_images[:8], train_labels[:8]):
    ax.imshow(image, cmap="gray")
    ax.set_title("".join(map(str, digits)))
    ax.axis("off")
plt.suptitle("Synthetic five-digit MNIST examples")
plt.tight_layout()
plt.show()


## 4. Normalize images and build a CNN

The network outputs **five probability distributions over the ten digit classes**, rather than five scaled numbers. The final shape is `(batch, 5, 10)`.

- **Loss:** sparse categorical cross-entropy, applied to the five digit positions.
- **Training metric:** digit-level classification accuracy, averaged over positions.
- **Evaluation metric:** exact-sequence accuracy (all five positions correct), calculated separately after prediction.

In [ ]:
def normalize_images(images):
    """Convert uint8 images to float32 grayscale tensors in [0, 1]."""
    return (images.astype("float32") / 255.0)[..., np.newaxis]


X_train = normalize_images(train_images)
X_valid = normalize_images(valid_images)
X_test = normalize_images(test_images)

# Integer class labels in 0..9; do NOT divide them by 255.
Y_train = train_labels
Y_valid = valid_labels
Y_test = test_labels

print(f"Model inputs: {X_train.shape}; targets: {Y_train.shape}")


In [ ]:
def build_model():
    """CNN predicting one categorical digit label for each of 5 positions."""
    model = keras.Sequential(
        [
            layers.Input(shape=(22, 90, 1)),
            layers.Conv2D(
                64, kernel_size=5, strides=2, padding="same",
                kernel_regularizer=regularizers.l2(1e-4),
            ),
            layers.LeakyReLU(negative_slope=0.2),
            layers.Dropout(0.25),
            layers.Conv2D(
                128, kernel_size=5, strides=2, padding="same",
                kernel_regularizer=regularizers.l2(1e-4),
            ),
            layers.LeakyReLU(negative_slope=0.2),
            layers.Dropout(0.25),
            layers.Flatten(),
            layers.Dense(DIGITS_PER_IMAGE * NUM_CLASSES),
            layers.Reshape((DIGITS_PER_IMAGE, NUM_CLASSES)),
            layers.Softmax(axis=-1),
        ],
        name="five_digit_cnn",
    )
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=2e-4, beta_1=0.5),
        loss=keras.losses.SparseCategoricalCrossentropy(),
        metrics=[keras.metrics.SparseCategoricalAccuracy(name="digit_accuracy")],
    )
    return model


model = build_model()
model.summary()


## 5. Train and visualize learning curves

Early stopping restores the weights from the epoch with the best validation loss. Adjust `EPOCHS`, `TRAIN_SAMPLES`, and `BATCH_SIZE` in the configuration cell for faster experiments.

In [ ]:
history = model.fit(
    X_train,
    Y_train,
    validation_data=(X_valid, Y_valid),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    shuffle=True,
    callbacks=[
        keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=4, restore_best_weights=True
        )
    ],
    verbose=1,
)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history["loss"], label="Train")
axes[0].plot(history.history["val_loss"], label="Validation")
axes[0].set(title="Cross-entropy loss", xlabel="Epoch", ylabel="Loss")
axes[0].legend()

axes[1].plot(history.history["digit_accuracy"], label="Train")
axes[1].plot(history.history["val_digit_accuracy"], label="Validation")
axes[1].set(title="Per-digit accuracy", xlabel="Epoch", ylabel="Accuracy")
axes[1].legend()

plt.tight_layout()
plt.show()


## 6. Evaluate on the held-out MNIST test split

**Digit accuracy** measures how often an individual digit position is classified correctly. **Exact-sequence accuracy** is stricter: all five digits in an image must be correct. Do not confuse the two metrics.

In [ ]:
test_metrics = model.evaluate(X_test, Y_test, verbose=0, return_dict=True)
probabilities = model.predict(X_test, batch_size=BATCH_SIZE, verbose=0)
predicted_digits = probabilities.argmax(axis=-1)

per_digit_accuracy = np.mean(predicted_digits == Y_test)
sequence_accuracy = np.mean(np.all(predicted_digits == Y_test, axis=1))

print(f"Test loss:            {test_metrics['loss']:.4f}")
print(f"Test per-digit acc.:  {per_digit_accuracy:.2%}")
print(f"Test exact-match acc: {sequence_accuracy:.2%}")


In [ ]:
indices = [4, 6, 11, 19, 27, 35, 42, 50]
fig, axes = plt.subplots(2, 4, figsize=(14, 4))

for ax, index in zip(axes.flat, indices):
    true_number = "".join(map(str, Y_test[index]))
    predicted_number = "".join(map(str, predicted_digits[index]))
    is_correct = true_number == predicted_number

    ax.imshow(test_images[index], cmap="gray")
    ax.set_title(
        f"True: {true_number} | Pred: {predicted_number}",
        color="green" if is_correct else "red",
        fontsize=10,
    )
    ax.axis("off")

plt.suptitle("Five-digit recognition: test-set predictions")
plt.tight_layout()
plt.show()


## Notes and limitations

- Examples are **synthetically composed from MNIST digits**, not photographs of naturally written multi-digit numbers.
- The fixed crop (`22 × 18` per digit) may clip some handwriting strokes; it is preserved from the earlier notebook for continuity.
- Digits are horizontally concatenated without realistic handwriting spacing or overlap.
- Results should be reported only after running the notebook. No unverified accuracy values are claimed.